In [ ]:
# --- Step 1: 设置环境 ---

# Fork 项目
!git clone https://github.com/Cl0udTide/happy-llm
%cd happy-llm

# 安装所有依赖
!git checkout my-experiments
%cd ./my_experiments/_3_training_pipeline
%pip install -r ./requirements.txt

# --- 登录 SwanLab ---
import swanlab
swanlab.login()

In [ ]:
# ======================================================
# Step 2: 数据准备
# ======================================================
from datasets import load_dataset
import os

# 创建数据目录
os.makedirs("./data/wikipedia_cn", exist_ok=True)
os.makedirs("./data/alpaca_gpt4_zh", exist_ok=True)

# 下载并保存 Wikipedia 数据
print("正在下载 Wikipedia-CN 数据集...")
full_wiki_dataset = load_dataset("pleisto/wikipedia-cn-20230720-filtered", split="train")
print(f"完整维基百科数据集大小: {len(full_wiki_dataset)} 条")

subset_size = len(full_wiki_dataset) // 6
print(f"将使用其中六分之一的数据: {subset_size} 条")
# 使用 .select() 方法来创建一个子集
subset_wiki_dataset = full_wiki_dataset.select(range(subset_size))

subset_wiki_dataset.to_json("./data/wikipedia_cn/pretrain_data_subset.jsonl")
print("Wikipedia-CN 的六分之一子集保存成功！")

# 下载并保存 Alpaca 数据
print("\n正在下载 Alpaca-GPT4-ZH 数据集...")
alpaca_dataset = load_dataset("c-s-ale/alpaca-gpt4-data-zh", split="train")
alpaca_dataset.to_json("./data/alpaca_gpt4_zh/sft_data.jsonl")
print("Alpaca-GPT4-ZH 数据集保存成功！")

In [ ]:
# ======================================================
# Step 3: 启动预训练
# ======================================================

!python pretrain.py \
    --out_dir "./outputs/tiny_llama_pretrained_wiki_swanlab" \
    --data_path "./data/wikipedia_cn/pretrain_data_subset.jsonl" \
    --use_swanlab \
    --epochs 1 \
    --batch_size 4 \
    --accumulation_steps 8 \
    --learning_rate 3e-4 \
    --dtype "float16" \
    --gpus "0"

Traceback (most recent call last):
  File "e:\资料\课程资料\大三上\happy-llm\my_experiments\_3_training_pipeline\pretrain.py", line 15, in <module>
    from .._2_llama_implementation.model import Transformer, ModelConfig
ImportError: attempted relative import with no known parent package


In [ ]:
# ======================================================
# Step 4: 启动监督微调 (SFT)
# ======================================================

!python finetune.py \
    --base_model_path "./outputs/tiny_llama_pretrained_wiki/pretrain_256_4_8192.pth" \
    --out_dir "./outputs/tiny_llama_sft_alpaca" \
    --data_path "./data/alpaca_gpt4_zh/sft_data.jsonl" \
    --epochs 1 \
    --batch_size 2 \
    --accumulation_steps 8 \
    --learning_rate 2e-5 \
    --dtype "float16" \
    --gpus "0"